# LLM Architectures Explained: Encoder-Decoder Architecture (Part 4)

Code from the article [LLM Architectures Explained: Encoder-Decoder Architecture (Part 4)](https://medium.com/@vipra_singh/llm-architectures-explained-encoder-decoder-architecture-part-4-b96ace71394c), in the order it appears there. Every code cell below was run from top to bottom; the outputs are from that run. Cells marked **Added for this repository** are not in the article: they download data or set things up so the notebook runs on a fresh machine.

## Installing the libraries

Shell command from the article (the repository's requirements.txt installs the same packages):

```python
pip install torch spacy datasets sacrebleu
python -m spacy download de_core_news_sm
python -m spacy download en_core_web_sm
```

## Importing the libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import spacy
import sacrebleu
from collections import Counter, defaultdict, namedtuple
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

## Tokenization

In [2]:
spacy_de = spacy.load('de_core_news_sm')
spacy_en = spacy.load('en_core_web_sm')
def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)][::-1]
def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]

## Fields and vocabulary

In [3]:
class Vocab:
    def __init__(self, itos):
        self.itos = itos                                                   # index -> token
        self.stoi = defaultdict(int, {tok: i for i, tok in enumerate(itos)})  # token -> index; unknown -> 0 ('<unk>')
    def __len__(self):
        return len(self.itos)

class Field:
    def __init__(self, tokenize, init_token, eos_token, lower=False):
        self.tokenize, self.init_token, self.eos_token, self.lower = tokenize, init_token, eos_token, lower
        self.unk_token, self.pad_token = '<unk>', '<pad>'

    def preprocess(self, text):
        tokens = self.tokenize(text)
        return [tok.lower() for tok in tokens] if self.lower else tokens

    def build_vocab(self, texts, min_freq=1):
        counts = Counter(tok for text in texts for tok in self.preprocess(text))
        specials = [self.unk_token, self.pad_token, self.init_token, self.eos_token]
        words = sorted(tok for tok, n in counts.items() if n >= min_freq and tok not in specials)
        self.vocab = Vocab(specials + words)

    def numericalize(self, text):
        tokens = [self.init_token] + self.preprocess(text) + [self.eos_token]
        return torch.tensor([self.vocab.stoi[tok] for tok in tokens])

In [4]:
SRC = Field(tokenize=tokenize_de, init_token='<sos>', eos_token='<eos>', lower=True)
TRG = Field(tokenize=tokenize_en, init_token='<sos>', eos_token='<eos>', lower=True)

## Loading and building the vocabulary

In [5]:
dataset = load_dataset('bentrevett/multi30k')
train_data, valid_data, test_data = dataset['train'], dataset['validation'], dataset['test']
SRC.build_vocab(train_data['de'], min_freq=2)
TRG.build_vocab(train_data['en'], min_freq=2)

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Batching

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Batch = namedtuple('Batch', ['src', 'trg'])

def collate(examples):
    src = pad_sequence([SRC.numericalize(ex['de']) for ex in examples], padding_value=SRC.vocab.stoi[SRC.pad_token])
    trg = pad_sequence([TRG.numericalize(ex['en']) for ex in examples], padding_value=TRG.vocab.stoi[TRG.pad_token])
    return Batch(src.to(device), trg.to(device))  # each is [seq_len, batch_size]

BATCH_SIZE = 128
train_iterator = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
valid_iterator = DataLoader(valid_data, batch_size=BATCH_SIZE, collate_fn=collate)
test_iterator = DataLoader(test_data, batch_size=BATCH_SIZE, collate_fn=collate)

## Encoder

In [7]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src_len, batch_size]
        embedded = self.dropout(self.embedding(src)) # [src_len, batch_size, emb_dim]
        outputs, hidden = self.rnn(embedded)
        return hidden

## Decoder

In [8]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden):
        # input = [batch_size]
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.dropout(self.embedding(input)) # [1, batch_size, emb_dim]
        output, hidden = self.rnn(embedded, hidden) # [1, batch_size, hid_dim], [n_layers, batch_size, hid_dim]
        prediction = self.fc_out(output.squeeze(0)) # [batch_size, output_dim]
        return prediction, hidden

## Seq2Seq model

In [9]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden = self.encoder(src)

        input = trg[0, :]
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs

## Training loop

In [10]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for i, batch in enumerate(iterator):
        src = batch.src
        trg = batch.trg

        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

## Evaluation

In [11]:
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(iterator):
            src = batch.src
            trg = batch.trg
            output = model(src, trg, 0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

## Model, optimizer and loss

In [12]:
INPUT_DIM = len(SRC.vocab)
OUTPUT_DIM = len(TRG.vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Seq2Seq(enc, dec, device).to(device)
optimizer = optim.Adam(model.parameters())
TRG_PAD_IDX = TRG.vocab.stoi[TRG.pad_token]
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

## Training

In [13]:
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_iterator, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_iterator, criterion)

    print(f'Epoch: {epoch+1}')
    print(f'Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')

Epoch: 1
Train Loss: 4.916 | Val. Loss: 4.839


Epoch: 2
Train Loss: 4.256 | Val. Loss: 4.535


Epoch: 3
Train Loss: 3.933 | Val. Loss: 4.316


Epoch: 4
Train Loss: 3.675 | Val. Loss: 4.172


Epoch: 5
Train Loss: 3.500 | Val. Loss: 4.145


Epoch: 6
Train Loss: 3.373 | Val. Loss: 4.059


Epoch: 7
Train Loss: 3.248 | Val. Loss: 4.037


Epoch: 8
Train Loss: 3.143 | Val. Loss: 4.038


Epoch: 9
Train Loss: 3.064 | Val. Loss: 4.017


Epoch: 10
Train Loss: 2.986 | Val. Loss: 3.945


## Translating and BLEU

In [14]:
def translate_sentence(sentence, src_field, trg_field, model, device, max_len=50):
    model.eval()
    tokens = [token.lower() for token in tokenize_de(sentence)]  # reversed, like the training data
    tokens = [src_field.init_token] + tokens + [src_field.eos_token]

    src_indexes = [src_field.vocab.stoi[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden = model.encoder(src_tensor)

    trg_indexes = [trg_field.vocab.stoi[trg_field.init_token]]
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
            break
    trg_tokens = [trg_field.vocab.itos[i] for i in trg_indexes]
    return trg_tokens[1:]

In [15]:
example = test_data[0]
print('German:   ', example['de'])
print('Reference:', example['en'])
print('Model:    ', ' '.join(translate_sentence(example['de'], SRC, TRG, model, device)[:-1]))

hypotheses, references = [], []
for example in test_data:
    tokens = translate_sentence(example['de'], SRC, TRG, model, device)
    hypotheses.append(' '.join(tok for tok in tokens if tok != TRG.eos_token))
    references.append(' '.join(TRG.preprocess(example['en'])))
print(f"Test BLEU: {sacrebleu.corpus_bleu(hypotheses, [references], tokenize='none', force=True).score:.2f}")

German:    Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Reference: A man in an orange hat starring at something.
Model:     a man with a hat hat is looking at something .


Test BLEU: 15.72
